In [1]:
import os
os.chdir('./proj2_data')
os.getcwd()

'/home/tako/Kasetsart/statistics/project2/proj2_data'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### Load and Split Data

In [3]:
data = pd.read_csv('articles_handout.csv', sep=",")
data.head()

,text,label
0,boothroyd calls for lords speaker betty boothr...,1
1,stuart joins norwich from addicks norwich have...,2
2,tories urge change at the top tory delegates...,1
3,celtic unhappy over bulgaria date martin o nei...,2
4,green fear for transport ballot the green part...,1


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

In [5]:
print(X_train.shape)
print(X_test.shape)

(581,)
(146,)


### Functions

In [6]:
def evaluate_performance(y_true, y_pred, average_val='macro'):

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average=average_val)
    recall = recall_score(y_true, y_pred, average=average_val)
    f1 = f1_score(y_true, y_pred, average=average_val)

    from sklearn.metrics import confusion_matrix
    confusion = confusion_matrix(y_true, y_pred)

    return accuracy, precision, recall, f1, confusion

### Text Preprocessing

In [7]:
import nltk
custom_dir = '../../.devenv/state/venv/nltk_data' 
nltk.download('punkt_tab', download_dir=custom_dir)
nltk.download('stopwords', download_dir=custom_dir)
nltk.download('wordnet', download_dir=custom_dir)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     ../../.devenv/state/venv/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     ../../.devenv/state/venv/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     ../../.devenv/state/venv/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [8]:
def preprocess_text(text):
    import re, string
    from nltk.tokenize import word_tokenize 
    from nltk.corpus import stopwords 
    from nltk.stem import WordNetLemmatizer    
    
    word_tokens = word_tokenize(text)
    word_tokens = [word.lower() for word in word_tokens] 
    word_tokens = [re.sub(r'[^\w\s]', '', token) for token in word_tokens if re.sub(r'[^\w\s]', '', token)]
    corpus_stop_words = set(stopwords.words('english')) 
    word_tokens = [word for word in word_tokens if not word in corpus_stop_words] 
    lemmatizer = WordNetLemmatizer()
    word_tokens = [lemmatizer.lemmatize(word) for word in word_tokens]
    return word_tokens

In [9]:
processed_X_train = [" ".join(preprocess_text(word)) for word in X_train]
processed_X_train

['mp demand budget leak answer minister asked explain budget detail printed london newspaper half hour gordon brown made speech tory said large chunk budget appeared leaked describe serious breach treasury confidentiality lib dems called common leader peter hain make statement said chancellor resigned leak told would brought speaker michael martin attention common tory frontbencher andrew tyrie mp demanded immediate ministerial statement measure clearly least apparently leaked evening standard raising point order said latest long line discourtesy house well breach confidentiality said hope unintentional planned would grave matter indeed previous labour chancellor resigned leaked budget hugh dalton resigned leaking detail 1947 budget journalist john carvel published london newspaper minute announced house common liberal democrat david law said serious matter said mr hain make statement thursday deputy speaker sylvia heal agreed concern said nothing could done immediately issue would bro

In [10]:
processed_X_test = [" ".join(preprocess_text(word)) for word in X_test]
processed_X_test

['zambia confident cautious zambia technical director kalusha bwalya confident cautious ahead cosafa cup final angola saturday lusaka bwalya said nothing short victory however bwalya warned side complacent want team comfortable sure victory going difficult game main aim game enjoy win zambia shown determination win final recalling nine foreignbased player however 41 yearold bwalya became oldest player appear competition played scored mauritius uncertain whether take field chipolopolo fan however cautious victory concert already scheduled match featuring country top musician side hoping win competition record third time keep trophy good chipolopolo first two edition regional tournament southern african nation 1997 1998 prevented third straight win angola knocked zambian semifinal stage 1999 victory angola also marked first defeat 14 year zambia lusaka independence stadium saturday game played angola named four overseasbased player preliminary squad palancas negras unable secure release 

### Text Representation

In [11]:
# Convert nested list to word2vec vectors

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

train_result = tfidf.fit_transform(processed_X_train)
test_result = tfidf.transform(processed_X_test)
print(tfidf.get_feature_names_out())
print(train_result.toarray())

['00' '000' '0001' ... 'zone' 'zorro' 'zurich']
[[0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 ...
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.0882714 0.        ... 0.        0.        0.       ]]


In [13]:
feature_names = tfidf.get_feature_names_out()
tfidf_vectors = pd.DataFrame(train_result.toarray(), columns=feature_names)
print(tfidf_vectors)

      00       000  0001  000acre  000ayear  000m  000seat  000strong  000th  \
0    0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
1    0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
2    0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
3    0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
4    0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
..   ...       ...   ...      ...       ...   ...      ...        ...    ...   
576  0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
577  0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
578  0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
579  0.0  0.000000   0.0      0.0       0.0   0.0      0.0        0.0    0.0   
580  0.0  0.088271   0.0      0.0       0.0   0.0      0.0        0.0    0.0   

     000vote  ...  zimbabwe  zinc  zine

### Training Model and Evaluate Performance

In [14]:
# Classification model using X_train_vectors nd y_train as training dataset
# Use X_test_vectors as test dataset

In [15]:
def evaluate():
    y_predicted = clf.predict(test_result.toarray())
    # Evalute performance
    from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
    print(classification_report(y_test,y_predicted))
    
    print(y_predicted)
    
    accuracy, precision, recall, f1, confusion = evaluate_performance(y_test, y_predicted)
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 score:', f1)
    print('Confusion:\n', confusion)

In [16]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(tfidf_vectors, y_train)
evaluate()

              precision    recall  f1-score   support

           0       1.00      0.98      0.99        43
           1       1.00      1.00      1.00        54
           2       0.98      1.00      0.99        49

    accuracy                           0.99       146
   macro avg       0.99      0.99      0.99       146
weighted avg       0.99      0.99      0.99       146

[2 2 1 1 0 0 1 0 1 2 0 2 0 0 1 2 0 2 1 1 0 0 0 2 1 0 0 1 2 0 2 2 2 1 0 1 1
 0 0 2 0 0 2 2 2 1 0 1 2 1 2 0 2 1 0 2 2 2 2 1 0 0 1 1 2 1 2 2 2 1 2 1 2 1
 1 0 2 2 1 2 2 1 1 0 0 1 2 1 1 1 2 2 2 2 0 2 0 2 0 2 1 1 2 1 2 1 1 0 2 0 2
 1 0 0 2 0 0 2 1 1 0 1 2 0 1 1 0 2 1 1 1 2 1 2 1 0 1 1 0 1 1 1 0 0 1 1]
Accuracy: 0.9931506849315068
Precision: 0.9933333333333333
Recall: 0.9922480620155039
F1 score: 0.9927114280055457
Confusion:
 [[42  0  1]
 [ 0 54  0]
 [ 0  0 49]]


/nix/store/fqm2yfxsaigixmzyzbw9whdvnwi2mxjg-devenv-profile/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
